In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt

try:
    import snowflake.connector
except:
    ! pip install snowflake-connector-python
    import snowflake.connector

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:28:50.928799


#### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [4]:
def get_tier(flt_ecnl, dict_tiers):
    if flt_ecnl <= dict_tiers['A1']:
        return 'A1'
    elif flt_ecnl <= dict_tiers['A']:
        return 'A'
    elif flt_ecnl <= dict_tiers['B']:
        return 'B'
    elif flt_ecnl <= dict_tiers['C']:
        return 'C'
    elif flt_ecnl <= dict_tiers['D']:
        return 'D'
    else:
        return 'Decline'

#### Constants

In [5]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

# inst
list_str_inst = [
    # from ben: 2025-02-21
    'CURRENT',
    'SELF',
    # key words
    'CHIME-STRIDE',
    'CHIMEFINAL',
    # from dustin: 2025-02-24
    'SELF FIN',
    'SELF/LEAD',
    'SELFINC/LEAD',
    'SBNASELFLNDR',
    'SBNA SELF',
    'CHIME',
    'CLEO',
    'CLEO AI',
    'VARO',
    'ATLAS',
    'ATLCAPBKSELF',
    'POSSIBLE',
    'POSSIBLE FIN',
    'KIKOFF',
    'SUPER.COM',
    'STEP',
    'STEP MOBILE',
    'BRIGHT',
    'BRIGHT BLDR',
    'FIG TECH INC',
    'SELF/RENT',
    'SELFBILLSE',
    'PROGRESSRES',
    'FLEX',
    'FLEXFINANCE',
]

Project: 20250307-funded-trends
Task: 01_data_collection


#### Make output dir

In [6]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Connect to snowflake

In [7]:
# load 
str_filename = 'datascience_rsa_key.p8'
str_local_path = f'./{str_filename}'
with open(str_local_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend(),
    )
# convert to bytes
private_key = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
# connect to snowflake
conn = snowflake.connector.connect(
    user='datascience', 
    private_key=private_key,
    account='pfs', 
    warehouse='datascience',
    database='raw',
    schema='source_s3_scorehistory',
)

#### Load Gen 13 data

In [8]:
str_filename = 'df.gzip'
str_uri = f's3://20241112-simple-model-test/08_prep_data/{str_filename}'
df = pd.read_parquet(str_uri)
df['applicationdate__app'] = pd.to_datetime(df['applicationdate__app'])
df['dealerstampcreation__app'] = pd.to_datetime(df['dealerstampcreation__app'])
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
37865,5514485,2022-10-15 02:35:52.2455863,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Virginia,Independent,Virginia,False,...,0,0,5,0.111662,1.299997,0,1,1,0,0.000000
37866,5514970,2022-10-15 02:36:32.6396499,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,1,0,6,0.055245,1.295059,0,0,0,0,0.000000
37869,5515245,2022-10-15 02:36:53.5525392,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Texas,Franchise,Texas,False,...,1,0,0,0.087712,1.156227,0,0,0,0,0.000000
37870,5515340,2022-10-15 02:37:13.5377961,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,5,NaN,1.599743,0,0,0,0,NaN
37871,5515580,2022-10-15 02:37:34.1144830,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Kentucky,Franchise,Kentucky,False,...,1,1,4,NaN,1.363642,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94425,8420461,2024-11-26 02:27:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Pennsylvania,Independent,Pennsylvania,True,...,0,0,5,NaN,1.585118,0,0,0,0,0.000000
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94426,8420665,2024-11-26 02:37:11+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Ohio,Franchise,Ohio,True,...,1,0,2,NaN,1.189153,1,0,0,0,0.136585
94456,8421889,2024-11-26 05:10:18+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Georgia,Franchise,Georgia,True,...,1,0,0,0.206963,1.082309,0,0,0,1,0.000000


#### Load Gen 13 model monitoring data

In [9]:
str_filename = 'df.gzip'
str_uri = f's3://20250121-gen-13-model-monitoring/04_concatenate_files/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
df_tmp['applicationdate__app'] = pd.to_datetime(df_tmp['applicationdate__app'])
df_tmp['dealerstampcreation__app'] = pd.to_datetime(df_tmp['dealerstampcreation__app'])
# show
df_tmp

,accountid,request_datetime,response_model_name,file_name,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,BRIGHT_tag,BRIGHT BLDR_tag,FIG TECH INC_tag,SELF/RENT_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag
0,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,1,8427704104055851,8427704,10405585,1,Idaho,...,0,0,0,0,0,0,0,0,0,0
1,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,0,8427704104055860,8427704,10405586,0,Idaho,...,0,0,0,0,0,0,0,0,0,0
2,8427706,2024-11-27 00:00:38-07:00,PRESTIGE-GEN-XII,8427706/RB5u-2024-11-27-00:00:38,1,8427706104055881,8427706,10405588,1,Texas,...,0,0,0,0,0,0,0,0,0,0
3,8427707,2024-11-27 00:00:41-07:00,PRESTIGE-GEN-XII,8427707/q369-2024-11-27-00:00:41,1,8427707104055891,8427707,10405589,1,Ohio,...,0,0,0,0,0,0,0,0,0,0
4,8427708,2024-11-27 00:01:12-07:00,PRESTIGE-GEN-XII,8427708/8SAE-2024-11-27-00:01:12,1,8427708104055901,8427708,10405590,1,Alabama,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,8748389,2025-03-09 23:37:28-06:00,PRESTIGE-GEN-XIII,8748389/ws6p-2025-03-09-23:37:28,1,8748389107778621,8748389,10777862,1,California,...,0,0,0,0,0,0,0,0,2,1
9679,8748390,2025-03-09 23:39:37-06:00,PRESTIGE-GEN-XIII,8748390/bl2a-2025-03-09-23:39:37,1,8748390107778631,8748390,10777863,1,Oregon,...,0,0,0,0,0,0,0,0,1,1
9680,8748293,2025-03-09 23:40:12-06:00,PRESTIGE-GEN-XIII,8748293/0tM9-2025-03-09-23:40:12,1,8748293107777451,8748293,10777745,1,Washington,...,0,0,0,0,0,0,0,0,0,0
9681,8748391,2025-03-09 23:46:39-06:00,PRESTIGE-GEN-XIII,8748391/x8IZ-2025-03-09-23:46:39,1,8748391107778641,8748391,10777864,1,Hawaii,...,0,0,0,0,0,0,0,0,0,0


#### Find tags

In [10]:
list_cols = [col for col in df_tmp.columns if 'tag' in col and col != 'subjectage__ln']
list_cols.append('sum')
# drop
df_tmp.drop(list_cols, axis=1, inplace=True)
# show
df_tmp

,accountid,request_datetime,response_model_name,file_name,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,int_bad_6mo_closed_end__tu_pmthx,bktunew__tu,eqscrpd__tu,g213br__tu,g416sr__tu,in57sr__tu,rtltrd__tu,logscore__tu,adlogscr__tu,list_institutions
0,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,1,8427704104055851,8427704,10405585,1,Idaho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[ELAN CACS CC, CCB/LNDINGCL, SPS, CAPITAL ONE,..."
1,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,0,8427704104055860,8427704,10405586,0,Idaho,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[JPMCB CARD, BK OF AMER, ELAN CACS CC, JPMCB C..."
2,8427706,2024-11-27 00:00:38-07:00,PRESTIGE-GEN-XII,8427706/RB5u-2024-11-27-00:00:38,1,8427706104055881,8427706,10405588,1,Texas,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[VERIZON, CAPITAL ONE, DISCOVERBANK, APG FIN, ..."
3,8427707,2024-11-27 00:00:41-07:00,PRESTIGE-GEN-XII,8427707/q369-2024-11-27-00:00:41,1,8427707104055891,8427707,10405589,1,Ohio,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[CB/HELZBERG, MAZDA FIN SV, FEDLOAN, FEDLOAN, ..."
4,8427708,2024-11-27 00:01:12-07:00,PRESTIGE-GEN-XII,8427708/8SAE-2024-11-27-00:01:12,1,8427708104055901,8427708,10405590,1,Alabama,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[SETF/WOFC, EXETER FIN, CAPITAL ONE, EDFINANCI..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,8748389,2025-03-09 23:37:28-06:00,PRESTIGE-GEN-XIII,8748389/ws6p-2025-03-09-23:37:28,1,8748389107778621,8748389,10777862,1,California,...,1.0,1.0,0.478,-1.0,0.0,-3.0,4.0,1.336615,1.135945,"[WEBBNK/FHUT, LENDMARK, SETF/WOFC, MILITARYSTA..."
9679,8748390,2025-03-09 23:39:37-06:00,PRESTIGE-GEN-XIII,8748390/bl2a-2025-03-09-23:39:37,1,8748390107778631,8748390,10777863,1,Oregon,...,0.0,0.0,0.230,13327.0,0.0,0.0,4.0,-0.238404,-0.439075,"[MID OR FCU, CAPITAL ONE, FST PREMIER, FNWSE/O..."
9680,8748293,2025-03-09 23:40:12-06:00,PRESTIGE-GEN-XIII,8748293/0tM9-2025-03-09-23:40:12,1,8748293107777451,8748293,10777745,1,Washington,...,NaN,0.0,0.398,-1.0,0.0,-3.0,1.0,0.936187,0.735516,"[CAPITAL ONE, CAPITAL ONE, LES SCHWAB, ALLY FI..."
9681,8748391,2025-03-09 23:46:39-06:00,PRESTIGE-GEN-XIII,8748391/x8IZ-2025-03-09-23:46:39,1,8748391107778641,8748391,10777864,1,Hawaii,...,1.0,0.0,0.547,347.0,0.0,0.0,1.0,1.941404,1.740733,"[CAPITAL ONE, ONEMAIN, CITI, CAPITAL ONE, WEBB..."


#### Concatenate data

In [11]:
%%time

# concat
df = pd.concat([df, df_tmp])
# get date
df['request_datetime'] = df['request_datetime'].apply(
    lambda x: str(x)[:10],
)
# make datetime
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
# set dtype
df['accountid'] = df['accountid'].astype(int)
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
# rm dup rows
df.drop_duplicates(subset=['accountid','bitdebtor'], keep='last', inplace=True)
# show
df

CPU times: user 9.03 s, sys: 6.64 s, total: 15.7 s
Wall time: 15.7 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,totalincome__app,bktunew__tu,eqscrpd__tu,g213br__tu,g416sr__tu,in57sr__tu,rtltrd__tu,logscore__tu,adlogscr__tu,list_institutions
1,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9302,8727961,2025-03-09,PRESTIGE-GEN-XIII,NaN,1,1,Utah,Independent,Utah,True,...,6668.00,0.0,0.330,1770.0,5.0,-3.0,2.0,0.456887,0.256217,"[MTN AMER CU, MTN AMER CU, OPORTUNPROG, HC ROY..."
9304,8727962,2025-03-09,PRESTIGE-GEN-XIII,NaN,1,1,California,Independent,California,False,...,3700.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]
9305,8727965,2025-03-09,PRESTIGE-GEN-XIII,NaN,1,1,Texas,Franchise,Texas,False,...,7800.00,0.0,0.308,1441.0,0.0,0.0,0.0,0.283155,0.082485,"[DPT ED/AIDV, DPT ED/AIDV, FLEX]"
9299,8727960,2025-03-09,PRESTIGE-GEN-XIII,NaN,0,0,Alaska,Franchise,Alaska,False,...,11250.00,0.0,0.260,-1.0,6.0,-3.0,1.0,-0.052747,-0.253418,"[CREDITACPT, FETTIFHT/WEB, CHIME-STRIDE, NAFCO..."


#### Get funded apps

In [12]:
# query funded
str_filename = 'query_funded.sql'
str_local_path = f'./query/{str_filename}'
str_query = open(str_local_path, 'r').read()

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# set as int
df_tmp['ACCOUNT_NUMBER'] = df_tmp['ACCOUNT_NUMBER'].astype(int)
# list
list_int_account = list(df_tmp['ACCOUNT_NUMBER'])
# save memory
del df_tmp
list_int_account = list(dict.fromkeys(list_int_account))
# len
int_n_funded = len(list_int_account)
print(f'There are {int_n_funded} funded apps')

There are 87750 funded apps


#### Subset to funded

In [13]:
%%time

df['funded'] = df['accountid'].apply(
    lambda x: 1 if x in list_int_account else 0,
)
# subset
df = df[df['funded'] == 1].copy()
# show
df

CPU times: user 2min 26s, sys: 457 ms, total: 2min 27s
Wall time: 2min 27s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,bktunew__tu,eqscrpd__tu,g213br__tu,g416sr__tu,in57sr__tu,rtltrd__tu,logscore__tu,adlogscr__tu,list_institutions,funded
1,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
0,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
42,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
41,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,8705652,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Michigan,Franchise,Michigan,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[SANTANDER, CHIME-STRIDE, DPT ED/AIDV]",1
3836,8676945,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Pennsylvania,Independent,New Jersey,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[HUD TITLE I, BANKAMERICA, CAPITAL ONE, CU OF ...",1
3763,8712238,2025-03-07,PRESTIGE-GEN-XIII,NaN,0,0,Oklahoma,Franchise,Kansas,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[WEBBANK/OM, JPMCB CARD, TARGET/TD, CBW, CCB/S...",1
3702,8605449,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Ohio,Franchise,Ohio,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[CANTO SCHOOL, CANTO SCHOOL, CAPITAL ONE, CHIM...",1


#### Get DLv2 apps

In [14]:
# query funded
str_filename = 'query_direct.sql'
str_local_path = f'./query/{str_filename}'
str_query = open(str_local_path, 'r').read()

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# set as int
df_tmp['ACCOUNTID'] = df_tmp['ACCOUNTID'].astype(int)
# list
list_int_account = list(df_tmp['ACCOUNTID'])
# save memory
del df_tmp
list_int_account = list(dict.fromkeys(list_int_account))
# len
int_n_direct = len(list_int_account)
print(f'There are {int_n_direct} direct apps')

There are 71515 direct apps


#### Subset to indirect

In [15]:
%%time

df['direct'] = df['accountid'].apply(
    lambda x: 1 if x in list_int_account else 0,
)
# subset
df = df[df['direct'] == 0].copy()
# show
df

CPU times: user 48.2 s, sys: 702 ms, total: 48.9 s
Wall time: 48.9 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,eqscrpd__tu,g213br__tu,g416sr__tu,in57sr__tu,rtltrd__tu,logscore__tu,adlogscr__tu,list_institutions,funded,direct
1,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
0,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
42,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
41,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,8705652,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Michigan,Franchise,Michigan,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[SANTANDER, CHIME-STRIDE, DPT ED/AIDV]",1,0
3836,8676945,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Pennsylvania,Independent,New Jersey,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[HUD TITLE I, BANKAMERICA, CAPITAL ONE, CU OF ...",1,0
3763,8712238,2025-03-07,PRESTIGE-GEN-XIII,NaN,0,0,Oklahoma,Franchise,Kansas,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[WEBBANK/OM, JPMCB CARD, TARGET/TD, CBW, CCB/S...",1,0
3702,8605449,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Ohio,Franchise,Ohio,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[CANTO SCHOOL, CANTO SCHOOL, CAPITAL ONE, CHIM...",1,0


#### Create Chime tags

In [16]:
df['list_institutions'] = df['str_institution__tu_pmthx'].apply(
    lambda x: eval(x.replace('nan','None')),
)
df['list_institutions'] = df['list_institutions'].apply(
    lambda x: [] if x is None else x,
)
list_str_col_new = []
for str_inst in tqdm(list_str_inst):
    str_col_new = f'{str_inst}_tag'
    df[str_col_new] = df['list_institutions'].apply(
        lambda x: 1 if str_inst in x else 0,
    )
    list_str_col_new.append(str_col_new)
# sum of tags
df['sum'] = df[list_str_col_new].sum(axis=1)
# has a credit builder tag
df['has_inst_tag'] = df['sum'].apply(
    lambda x: 1 if x > 0 else 0,
)
flt_mn = df['has_inst_tag'].mean()
print(f'Proportion has tag: {flt_mn:0.4f}')

100%|██████████| 29/29 [00:01<00:00, 18.51it/s]


Proportion has tag: 0.2179


#### Drop chime tags that don't show up

In [17]:
list_cols = []
for col in tqdm(list_str_col_new):
    # get sum
    int_sum = df[col].sum()
    # logic
    if int_sum == 0:
        list_cols.append(col)
    else:
        pass
int_len = len(list_cols)
print(f'There were {int_len} chime institutions not in the data')
# drop
df.drop(list_cols, axis=1, inplace=True)
# show
df

100%|██████████| 29/29 [00:00<00:00, 7485.68it/s]


There were 13 chime institutions not in the data


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,SUPER.COM_tag,STEP MOBILE_tag,BRIGHT_tag,FIG TECH INC_tag,SELFBILLSE_tag,PROGRESSRES_tag,FLEX_tag,FLEXFINANCE_tag,sum,has_inst_tag
1,5714239,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,0,0,0,0,0,0,0,0,0
2,5713063,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,0,0,0,0,0,0
0,5702434,2021-07-26,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,0,0,0,0,0,0
42,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,0,0,0,0
41,5704330,2021-07-27,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Idaho,Franchise,Idaho,True,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3550,8705652,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Michigan,Franchise,Michigan,False,...,0,0,0,0,0,0,0,0,1,1
3836,8676945,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Pennsylvania,Independent,New Jersey,True,...,0,0,0,0,0,0,0,0,0,0
3763,8712238,2025-03-07,PRESTIGE-GEN-XIII,NaN,0,0,Oklahoma,Franchise,Kansas,True,...,0,0,0,0,0,0,0,0,0,0
3702,8605449,2025-03-07,PRESTIGE-GEN-XIII,NaN,1,1,Ohio,Franchise,Ohio,True,...,0,0,0,0,0,0,0,0,3,1


#### Engineer

In [18]:
# payment hx
df['ENG-wtd_avg'] = df['flt_wtd_avg_open__tu_pmthx'].fillna(df['flt_wtd_avg_closed__tu_pmthx'])

In [19]:
# ltv
df['ENG-loan_to_value'] = df['amtfinanced__app'] / df['bookvalue__app']

In [20]:
# bk
df['ENG-bk'] = df['intopenbktype__app'].apply(
    lambda x: 1 if pd.notnull(x) else 0,
)

#### Convert non-numeric to string

In [21]:
for col in tqdm(df.columns):
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 2803/2803 [00:17<00:00, 164.12it/s]  


#### Write to s3

In [22]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 34.8 s, sys: 424 ms, total: 35.2 s
Wall time: 35.2 s
